Dataset 라이브러리
- KLUE 데이터셋 주제 분류 태스크인 YNAT 데이터셋
- 현재 환경에 datasets 설치: pip install datasets==2.20.0
- 학습 데이터 45,678건 / 검증 데이터 9,107건
- 'guid', 'title', 'label', 'url', 'date' 5개 컬럼으로 구성
- label 분류를 위한 정답 값으로 IT 과학 0/ 경제 1/ 사회 2/ 생활문화 3/ 세계 4/ 스포츠 5/ 정치 6 구성됨

Transformers
- NumPy 1.26.x, PyTorch 2.2.2, Transformers 4.44.2 버전

Tokenizer
- BertTokenize: KLUE 데이터셋으로 학습한 klue/bert-base 모델 사용
- tokenizer 인코딩: 문자열 토큰화 작업 -> 토큰을 입력 식별자로 변환
- tokenizer 디코딩: 

In [1]:
# 시스템 경로
import sys
print(sys.executable)
print(sys.version)

# transformer 설치 확인
import torch
import transformers

print(f"Transformers 버전: {transformers.__version__}")
print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 디바이스: {'cuda' if torch.cuda.is_available() else 'cpu'}")

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/bin/python
3.11.9 (v3.11.9:de54cf5be3, Apr  2 2024, 07:12:50) [Clang 13.0.0 (clang-1300.0.29.30)]
Transformers 버전: 4.44.2
PyTorch 버전: 2.2.2
사용 디바이스: cpu


In [2]:
# KLUE 데이터셋 주제 분류 태스크인 YNAT 데이터셋 load
from datasets import load_dataset

# klue/klue 경로 및 ynat 서브셋 지정
dataset = load_dataset("klue/klue", "ynat", trust_remote_code=True)

# 결과 확인
print(dataset)
print("\n첫 번째 샘플:")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9107
    })
})

첫 번째 샘플:
{'guid': 'ynat-v1_train_00000', 'title': '유튜브 내달 2일까지 크리에이터 지원 공간 운영', 'label': 3, 'url': 'https://news.naver.com/main/read.nhn?mode=LS2D&mid=shm&sid1=105&sid2=227&oid=001&aid=0008508947', 'date': '2016.06.30. 오전 10:36'}


In [3]:
# 비율로 데이터셋 불러오기
from datasets import load_dataset

ratio = 80
dataset = load_dataset(
    "klue/klue",
    "ynat",
    # path="csv",
    # data_files="sample.csv",
    split={
        "train": f"train[:{ratio}%]",
        "test": f"train[{ratio}%:]",
    },
    trust_remote_code=True
)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 36542
    })
    test: Dataset({
        features: ['guid', 'title', 'label', 'url', 'date'],
        num_rows: 9136
    })
})


In [ ]:
# KLUE 데이터셋으로 학습한 klue/bert-base 모델 사용 
from transformers import BertTokenizer

# 토크나이저를 다운로드하여 tokenizer 변수에 BertTokenizer 객체를 할당
model_name = "klue/bert-base"
tokenizer = BertTokenizer.from_pretrained(model_name)

# help 구문으로 tokenizer 변수에 담긴 Tokenizer 객체의 설명
# - BertTokenizer는 PretrainedTokenizer를 상속받고 있으며 워드피스에 기반하고 있다
# help(tokenizer) 

# tokenizer.vocab_size 학습한 사전 크기, tokenizer.get_vocab 토큰 사전 확인, tokenizer.special_tokens_map 새로운 특수 토큰
print(tokenizer.vocab_size)
print(tokenizer.get_vocab)
print(tokenizer.special_tokens_map)

32000
<bound method BertTokenizer.get_vocab of BertTokenizer(name_or_path='klue/bert-base', vocab_size=32000, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}>
{'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mas

/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
# 토큰화 프로세스: 인코딩
# Tokenizer: KLUE 데이터셋으로 학습한 klue/bert-base 모델 사용
from transformers import BertTokenizer
import warnings

# Transformers 의 경고(FutureWarning) 메시지 숨기기
warnings.filterwarnings("ignore", category=FutureWarning)

# pretrained 모델 다운로드
model_name = "klue/bert-base"
tokenizer = BertTokenizer.from_pretrained(model_name)

sentence = "안녕하세요. 이거 테스트 문장입니다."

# tokenize 메서드: 문자열 토큰리스트 반환
token1 = tokenizer.tokenize(sentence) 
print(f"tokenize 메서드: {token1}")

# convert_tokens_to_ids 메서드: 토큰을 정수 리스트인 토큰 아이디로 반환
ids1 = tokenizer.convert_tokens_to_ids(token1)
print(f"convert_tokens_to_ids 메서드: {ids1}")

# tokenizer 객체: input_ids, token_type_ids, attention_mask 반환
# input_ids: 문장 식별자 ID, 처음과 끝에 각각 2와3 식별자가 붙어 있음을 확인(klue/bert-base 모델에서 시작/종료 [CLS]/[SEP] 특수 번호 포함)
# token_type_ids: 문자열 시쿼스 번호, 입력으로 들어온 텍스트 시퀀스(그룹)의 번호
# attention_mask: 의미가 있으면 1, 없으면 패딩 0, 모델이 연산(Self-Attention)을 할때 진짜 학습/계산해야 할 텍스트 인지(1)/아니면(0) 또는 단순 길이 맞추기용([PAD])
ids2 = tokenizer(sentence)
print(f"tokenizer 객체:{ids2}")

tokenize 메서드: ['안녕', '##하', '##세요', '.', '이거', '테스트', '문장', '##입니다', '.']
convert_tokens_to_ids 메서드: [5891, 2205, 5971, 18, 4647, 7453, 6265, 12190, 18]
tokenizer 객체:{'input_ids': [2, 5891, 2205, 5971, 18, 4647, 7453, 6265, 12190, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [6]:
# 토큰화 프로세스: 디코딩

# ids1([CLS]/[SEP] 특수 토큰이 없는 id(정수) 리스트)
decoded_string1 = tokenizer.decode(ids1)
print(f"decode 메서드1: {decoded_string1}")

# ids2([CLS]/[SEP] 특수 토큰이 포함된 id(정수) 리스트), 
# 토크나이저 객체를 직접 호출해서 얻은 정석적인 input_ids를 디코딩한 것
decoded_string2 = tokenizer.decode(ids2['input_ids'])
print(f"decode 메서드2: {decoded_string2}")

# ids3([CLS]/[SEP] 특수 토큰이 포함된 id(정수) 리스트)
# 토크나이저 객체를 직접 호출해서 얻은 정석적인 input_ids를 디코딩한 것
# input_ids + skip_special_tokens=True (모델 내부 연산용 특수 토큰 [CLS]/[SEP]/[PAD] 등을 제거하고 순수 사용자 텍스트만 추출)
decoded_string3 = tokenizer.decode(ids2['input_ids'], skip_special_tokens=True)
print(f"decode 메서드3: {decoded_string3}")

decode 메서드1: 안녕하세요. 이거 테스트 문장입니다.
decode 메서드2: [CLS] 안녕하세요. 이거 테스트 문장입니다. [SEP]
decode 메서드3: 안녕하세요. 이거 테스트 문장입니다.
